# Chapter 3 Lab — Prompting Techniques: Zero/One/Few-Shot, Chain-of-Thought, ReAct, and Constrained Output

**Estimated time:** 25 minutes (Part A) + 10 minutes (Part B, optional)

- **Part A — offline, read-and-run.** A mock LLM and a mock search tool show the *shape* of a ReAct loop and of schema-constrained output. No API key, no network.
- **Part B — bring your own key (optional).** The same prompting techniques against a **live hosted model**, so you can see the difference zero-shot, one-shot, few-shot and chain-of-thought actually make in the model's output. Works with a free NVIDIA API key from [build.nvidia.com](https://build.nvidia.com) (NIM, OpenAI-compatible) or an OpenAI key. Standard library only — no SDK to install.

> Why a mock in Part A? So the loop logic is visible and deterministic, and so the lab runs with no account anywhere. Part B is where the technique differences become real.

## Part A · 1 — A mock search tool

In [ ]:
# A tiny knowledge base our 'search' tool can hit.
KB = {
  'nvidia nim': 'NVIDIA NIM is a set of inference microservices for deploying models.',
  'triton': 'Triton Inference Server serves models with dynamic batching and multiple backends.',
  'tensorrt-llm': 'TensorRT-LLM compiles optimized inference engines for LLMs on NVIDIA GPUs.'}
def search_tool(q):
    q=q.lower()
    hits=[v for k,v in KB.items() if k in q or any(t in q for t in k.split())]
    return hits[0] if hits else 'no result'

## Part A · 2 — A ReAct loop with a deterministic stand-in for the model

In [ ]:
# A deterministic 'policy' standing in for an LLM: Thought -> Action -> (Observation) -> Answer.
def mock_policy(question, scratch):
    if not scratch:
        return {'type':'action','tool':'search','input':question}
    obs = scratch[-1]['observation']
    return {'type':'final','answer':obs}

def react(question, policy=mock_policy, max_steps=4):
    scratch=[]
    for step in range(max_steps):
        act = policy(question, scratch)
        print(f'Thought {step}: need info' if act['type']=='action' else f'Thought {step}: have enough')
        if act['type']=='final':
            return act['answer']
        obs = search_tool(act['input'])
        print(f'  Action: search({act["input"]!r}) -> Observation: {obs[:60]}...')
        scratch.append({'action':act,'observation':obs})
    return 'gave up'
answer = react('What is NVIDIA NIM?')
print('\nFinal:', answer)

## Part A · 3 — Constrain the final answer to a JSON schema

In [ ]:
SCHEMA = {'answer': str, 'source': str, 'confidence': float}
def validate(obj, schema):
    errs=[]
    for key, typ in schema.items():
        if key not in obj: errs.append(f'missing {key}')
        elif not isinstance(obj[key], typ): errs.append(f'{key} must be {typ.__name__}')
    return errs

good = {'answer': answer, 'source':'KB', 'confidence':0.9}
bad  = {'answer': answer, 'confidence':'high'}  # wrong type + missing field
print('good ->', validate(good, SCHEMA) or 'VALID')
print('bad  ->', validate(bad, SCHEMA))

## Part B — Bring your own key: the same techniques on a live model

Set **one** environment variable before running the next cell, or paste a key into `API_KEY` for the session (do not commit it):

| Provider | Env var | Get a key | Endpoint / default model |
|---|---|---|---|
| NVIDIA NIM (recommended, free tier) | `NVIDIA_API_KEY` | https://build.nvidia.com → any model → *Get API Key* | `integrate.api.nvidia.com/v1` · `meta/llama-3.1-8b-instruct` |
| OpenAI | `OPENAI_API_KEY` | https://platform.openai.com | `api.openai.com/v1` · `gpt-4o-mini` |

Both speak the same OpenAI-compatible chat-completions API, so one `chat()` function covers both. If no key is found, the cells below print a notice and skip — the rest of the notebook still runs.

In [ ]:
import os, json, urllib.request, urllib.error

API_KEY = ''   # optional: paste a key here for this session only (never commit it)
PROVIDERS = {
    'nvidia': {'env': 'NVIDIA_API_KEY', 'url': 'https://integrate.api.nvidia.com/v1/chat/completions', 'model': 'meta/llama-3.1-8b-instruct'},
    'openai': {'env': 'OPENAI_API_KEY', 'url': 'https://api.openai.com/v1/chat/completions',        'model': 'gpt-4o-mini'},
}
PROVIDER, KEY = None, None
for name, cfg in PROVIDERS.items():
    k = API_KEY or os.environ.get(cfg['env'])
    if k: PROVIDER, KEY = name, k; break

LIVE = PROVIDER is not None
print('Live model:', f"{PROVIDER} / {PROVIDERS[PROVIDER]['model']}" if LIVE else 'NOT configured (set NVIDIA_API_KEY or OPENAI_API_KEY to run Part B)')

def chat(messages, temperature=0.0, max_tokens=300, model=None):
    """Minimal OpenAI-compatible chat call using only the standard library."""
    cfg = PROVIDERS[PROVIDER]
    body = json.dumps({'model': model or cfg['model'], 'messages': messages,
                       'temperature': temperature, 'max_tokens': max_tokens}).encode()
    req = urllib.request.Request(cfg['url'], data=body, method='POST', headers={
        'Authorization': f'Bearer {KEY}', 'Content-Type': 'application/json', 'Accept': 'application/json'})
    try:
        with urllib.request.urlopen(req, timeout=60) as r:
            return json.load(r)['choices'][0]['message']['content'].strip()
    except urllib.error.HTTPError as e:
        return f'[HTTP {e.code}] {e.read().decode()[:200]}'

### B · 1 — Zero-shot vs one-shot vs few-shot

Same task, three prompts. The task is deliberately one where the *output format* matters: classify a support ticket into one of four labels and return **only** the label. Watch how the zero-shot answer tends to add words or pick a label that is not in the list, one example fixes the format, and three examples fix the borderline case.

In [ ]:
TICKETS = [
    'My invoice shows two charges for October but I only have one subscription.',
    'The dashboard loads but every chart says "no data" since this morning.',
    'How do I add a second admin to our workspace?',
]
LABELS = ['billing', 'bug', 'how-to', 'feature-request']
EXAMPLES = [
    ('I was charged after cancelling last month.', 'billing'),
    ('Export to CSV fails with a 500 error.', 'bug'),
    ('Can you add dark mode to the mobile app?', 'feature-request'),
]
SYSTEM = f'You are a support triage assistant. Classify the ticket into exactly one of: {", ".join(LABELS)}. Reply with the label only.'

def classify(ticket, n_shots):
    msgs = [{'role':'system','content':SYSTEM}]
    for text, label in EXAMPLES[:n_shots]:
        msgs += [{'role':'user','content':text}, {'role':'assistant','content':label}]
    msgs.append({'role':'user','content':ticket})
    return chat(msgs, max_tokens=20)

if LIVE:
    print(f'{"ticket":60s} | zero-shot          | one-shot           | few-shot (3)')
    for t in TICKETS:
        row = [classify(t, n) for n in (0, 1, 3)]
        print(f'{t[:58]:60s} | ' + ' | '.join(f'{r[:18]:18s}' for r in row))
    print('\nValid label?  zero/one/few:', [all(classify(t, n).strip().lower() in LABELS for t in TICKETS) for n in (0,1,3)])
else:
    print('Skipped (no API key).')

### B · 2 — Chain-of-thought: reasoning depth, not format

A small arithmetic-with-a-twist question. Direct prompting asks for the number; chain-of-thought asks the model to work it out step by step *before* answering. Compare the answers and the token counts — CoT costs more tokens and buys accuracy on multi-step questions, which is exactly the trade-off the exam asks about.

In [ ]:
Q = ('A training run uses 64 GPUs for 18 hours at $2.50 per GPU-hour. The team reruns it twice more with a 25% '
     'smaller dataset each time, which cuts wall-clock time proportionally. What is the total GPU cost of all three runs? Give a dollar figure.')
if LIVE:
    direct = chat([{'role':'user','content': Q + ' Answer with the number only.'}], max_tokens=30)
    cot    = chat([{'role':'user','content': Q + ' Think step by step, then give the final answer on its own last line.'}], max_tokens=400)
    print('DIRECT :', direct)
    print('\nCoT    :', cot)
    print('\nExpected: 64*18*2.50 = $2,880; then 0.75x = $2,160; then 0.75x again = $1,620; total $6,660')
    print(f'Approx. output tokens - direct: ~{len(direct.split())*1.3:.0f}, CoT: ~{len(cot.split())*1.3:.0f}')
else:
    print('Skipped (no API key).')

### B · 3 — ReAct with the live model as the policy

Now swap the deterministic `mock_policy` for the real model. The model must emit a JSON action (`search` or `final`); the loop from Part A runs it unchanged. This is the Part A loop with a real brain in it.

In [ ]:
REACT_SYSTEM = ('You answer questions using a search tool. At each step reply with ONE JSON object and nothing else: '
                'either {"type":"action","tool":"search","input":"<query>"} to look something up, '
                'or {"type":"final","answer":"<answer>"} when you have enough. Search once, then answer.')

def live_policy(question, scratch):
    msgs = [{'role':'system','content':REACT_SYSTEM}, {'role':'user','content':question}]
    for s in scratch:
        msgs.append({'role':'assistant','content':json.dumps(s['action'])})
        msgs.append({'role':'user','content':f'Observation: {s["observation"]}'})
    raw = chat(msgs, max_tokens=120)
    try:
        start, end = raw.find('{'), raw.rfind('}')
        return json.loads(raw[start:end+1])
    except Exception:
        return {'type':'final','answer':f'(could not parse action) {raw[:120]}'}

if LIVE:
    ans = react('What does TensorRT-LLM do?', policy=live_policy)
    print('\nFinal:', ans)
    print('schema check ->', validate({'answer': ans, 'source':'KB', 'confidence':0.8}, SCHEMA) or 'VALID')
else:
    print('Skipped (no API key).')

## Exercise (10 minutes, Part B)
1. In B·1, change the few-shot examples so none of them is a `billing` ticket. Does the billing ticket still classify correctly? That is the *coverage* question behind few-shot prompting.
2. In B·2, set `temperature=0.7` and run CoT three times. Do the intermediate steps change while the final answer stays? That is why **self-consistency** samples several chains and votes.
3. In B·3, remove the sentence "Search once, then answer." What does the model do with `max_steps=4`?

## Key takeaway
Zero/one/few-shot prompting fixes **format and borderline cases** by example; chain-of-thought adds **reasoning depth** at a token cost; ReAct adds **tool access**; schema-constrained decoding fixes **output format** at the boundary. Each targets a different failure mode — picking the wrong one burns tokens without fixing the real problem.